# 2.Ingesta Bronze - Almacenes

## Objetivo
Ingerir el archivo **almacenes.csv** desde la capa **Raw** de Azure Data Lake Storage Gen2 hacia la capa **Bronze**, almacenándolo en formato Delta y registrando metadatos técnicos de ingesta.

Se utiliza Managed Identity mediante las External Locations configuradas en Unity Catalog. La lectura inicial permite observar la inferencia de tipos de Spark y la segunda lectura aplica un esquema explícito definido para este dataset.

Este notebook es independiente y forma parte del proceso de ingesta Bronze del proyecto.

## 1. Configuración inicial

In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DateType, TimestampType, DecimalType
)

## 2. Parámetros de ejecución

In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.text("containerRaw", "raw")
dbutils.widgets.text("containerDataLake", "datalake")
dbutils.widgets.text("storageName", "stproyectofinalehs")
dbutils.widgets.text("catalogo", "dbw_proyecto_final_ehs")

containerRaw = dbutils.widgets.get("containerRaw")
containerDataLake = dbutils.widgets.get("containerDataLake")
storageName = dbutils.widgets.get("storageName")
catalogo = dbutils.widgets.get("catalogo")

esquema = "bronze"

ruta_raw = f"abfss://{containerRaw}@{storageName}.dfs.core.windows.net/"
ruta_bronze = f"abfss://{containerDataLake}@{storageName}.dfs.core.windows.net/bronze/"

print("Catálogo:", catalogo)
print("Esquema:", esquema)
print("Ruta Raw:", ruta_raw)
print("Ruta Bronze:", ruta_bronze)

## 3. Ingesta de almacenes

In [0]:
# Lectura inicial para observar los tipos inferidos por Spark
df_almacenes_inferido = spark.read\
    .option("header", True)\
    .option("inferSchema", True)\
    .csv(f"{ruta_raw}almacenes.csv")

df_almacenes_inferido.printSchema()

# Definición manual del esquema
almacenes_schema = StructType([
    StructField("almacen_id", IntegerType(), True),
    StructField("nombre_almacen", StringType(), True),
    StructField("ciudad", StringType(), True),
    StructField("region", StringType(), True),
    StructField("es_activo", IntegerType(), True)
])

# Lectura aplicando el esquema definido
df_almacenes = spark.read\
    .option("header", True)\
        .schema(almacenes_schema)\
    .csv(f"{ruta_raw}almacenes.csv")

df_almacenes = df_almacenes\
    .withColumn("_source_file", lit("almacenes.csv"))\
    .withColumn("_ingestion_timestamp", current_timestamp())

df_almacenes.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .option("path", f"{ruta_bronze}almacenes/")\
    .saveAsTable(f"{catalogo}.{esquema}.almacenes")

print("Almacenes:", df_almacenes.count())

## 4. Validación de la ingesta

In [0]:
# Validación final de la ingesta
nombre_tabla = f"{catalogo}.{esquema}.almacenes"
df_validacion = spark.table(nombre_tabla)

cantidad_obtenida = df_validacion.count()
cantidad_esperada = 20

print("Tabla validada:", nombre_tabla)
print("Registros esperados:", cantidad_esperada)
print("Registros obtenidos:", cantidad_obtenida)
print("Esquema:")
df_validacion.printSchema()

# Verificación de metadatos técnicos
columnas_requeridas = ['_source_file', '_ingestion_timestamp']
columnas_faltantes = [c for c in columnas_requeridas if c not in df_validacion.columns]

if columnas_faltantes:
    raise ValueError(f"Faltan columnas técnicas: {columnas_faltantes}")

if cantidad_obtenida != cantidad_esperada:
    raise ValueError(
        f"Ingesta no válida para almacenes: "
        f"se esperaban {cantidad_esperada} registros y se obtuvieron {cantidad_obtenida}."
    )

print("VALIDACIÓN EXITOSA: la tabla Bronze contiene el total esperado y sus metadatos técnicos.")

## 5. Resultado esperado

| Tabla | Registros esperados |
|---|---:|
| `almacenes` | 20 |

La tabla Bronze conserva los registros de origen, incluidos los valores nulos y las incidencias detectadas durante el perfilado. La depuración y las reglas de calidad se implementarán en Silver.

La tabla incluye las columnas técnicas `_source_file` y `_ingestion_timestamp`.

**Criterio de aceptación:** la última celda debe finalizar con `VALIDACIÓN EXITOSA`.